# Phase 0 — Data Download

**Notebook:** `notebooks/00_setup/02_download_data.ipynb`

This downloads the four datasets we'll use in Phase 0–2 of the thesis:

| # | Dataset | Source | Purpose | Approx size |
|---|---------|--------|---------|-------------|
| 1 | NSL-KDD | GitHub mirror | Tiny dataset for pipeline sanity checks | ~25 MB |
| 2 | CICIDS2017 (original) | Kaggle | Reproducibility / comparison to other papers | ~500 MB |
| 3 | CICIDS2017 (Engelen-improved) | Kaggle | **Primary dataset for experiments** | ~600 MB |
| 4 | UNSW-NB15 | Kaggle | Second real dataset for Phase 2 | ~200 MB |

Total ≈ 1.4 GB. Well within your 100 GB Drive budget.

### Prerequisites

1. Phase 0 folder structure already created (run `01_folder_setup.ipynb` first).
2. A **Kaggle API token**. To get one:
   - Go to https://www.kaggle.com/settings/account
   - Scroll to *API* section → click **Create New Token**
   - This downloads `kaggle.json`. Keep it handy — you'll upload it below.
   - It's free and takes 2 minutes. No paid Kaggle account needed.

### Design notes

- All downloads stage to `/content/data_stage/` (fast Colab SSD) and then copy to Drive.
- Re-running is safe: existing files are skipped, not re-downloaded.
- If any download fails, the cell prints **manual fallback instructions** — read them, place files in the indicated folder, re-run.

## 1. Mount Drive and set up paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/phd_thesis')

import os, shutil, zipfile, json
from pathlib import Path
from urllib.request import urlretrieve
from tqdm.auto import tqdm

THESIS_ROOT = Path('/content/drive/MyDrive/phd_thesis')
RAW = THESIS_ROOT / 'data' / 'raw'
RAW.mkdir(parents=True, exist_ok=True)

STAGE = Path('/content/data_stage')
STAGE.mkdir(exist_ok=True)

def human_size(n: int) -> str:
    for unit in ['B', 'KB', 'MB', 'GB']:
        if n < 1024:
            return f'{n:.1f}{unit}'
        n /= 1024
    return f'{n:.1f}TB'

def disk_status():
    drv = shutil.disk_usage(THESIS_ROOT)
    loc = shutil.disk_usage('/content')
    print(f'Drive:  {human_size(drv.free)} free of {human_size(drv.total)}')
    print(f'Local:  {human_size(loc.free)} free of {human_size(loc.total)}')

disk_status()

Mounted at /content/drive
Drive:  54.7GB free of 100.0GB
Local:  205.4GB free of 225.8GB


## 2. Set up Kaggle API

Run this cell, then upload your `kaggle.json` file when prompted. It will be saved to Drive so you only do this **once** — future sessions reuse the saved token.

In [2]:
KAGGLE_DIR = Path.home() / '.kaggle'
KAGGLE_DIR.mkdir(exist_ok=True)
KAGGLE_JSON = KAGGLE_DIR / 'kaggle.json'

DRIVE_TOKEN = THESIS_ROOT / '.kaggle' / 'kaggle.json'
DRIVE_TOKEN.parent.mkdir(parents=True, exist_ok=True)

if DRIVE_TOKEN.exists():
    print('Found saved Kaggle token on Drive — reusing.')
    shutil.copy2(DRIVE_TOKEN, KAGGLE_JSON)
else:
    print('No saved token found. Upload your kaggle.json:')
    from google.colab import files
    uploaded = files.upload()
    name = list(uploaded.keys())[0]
    shutil.move(name, KAGGLE_JSON)
    shutil.copy2(KAGGLE_JSON, DRIVE_TOKEN)
    print(f'Token saved to {DRIVE_TOKEN} for future sessions.')

os.chmod(KAGGLE_JSON, 0o600)

# Install / verify Kaggle CLI
import subprocess
subprocess.run(['pip', 'install', '-q', 'kaggle'], check=True)

# Quick verification
result = subprocess.run(['kaggle', 'datasets', 'list', '-s', 'cicids', '--max-size', '1'],
                        capture_output=True, text=True)
if 'ref' in result.stdout.lower() or 'title' in result.stdout.lower():
    print('Kaggle API working ✓')
else:
    print('Kaggle API check inconclusive. Output:')
    print(result.stdout[:500])
    print(result.stderr[:500])

No saved token found. Upload your kaggle.json:


Saving kaggle.json to kaggle.json
Token saved to /content/drive/MyDrive/phd_thesis/.kaggle/kaggle.json for future sessions.
Kaggle API check inconclusive. Output:
No datasets found




## 3. Helpers

In [3]:
def download_url(url: str, dest: Path):
    """Download with progress. Skip if file already present."""
    if dest.exists() and dest.stat().st_size > 0:
        print(f'  [skip] {dest.name} already present ({human_size(dest.stat().st_size)})')
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    print(f'  [get]  {url}')
    pbar = tqdm(unit='B', unit_scale=True, miniters=1, desc=dest.name)
    def hook(blocks, block_size, total_size):
        if pbar.total is None and total_size > 0:
            pbar.total = total_size
        pbar.update(block_size)
    try:
        urlretrieve(url, dest, reporthook=hook)
    finally:
        pbar.close()
    return dest

def download_kaggle_dataset(slug: str, dest_dir: Path):
    """Download and unzip a Kaggle dataset by its slug, e.g. 'cicdatasets/cicids2017'."""
    dest_dir.mkdir(parents=True, exist_ok=True)
    if any(dest_dir.iterdir()):
        print(f'  [skip] {dest_dir} already populated')
        return dest_dir
    print(f'  [get]  kaggle datasets download -d {slug}')
    result = subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', slug, '-p', str(dest_dir), '--unzip'],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(f'  [FAIL] {result.stderr.strip()}')
        raise RuntimeError(result.stderr)
    print(f'  [done] extracted to {dest_dir}')
    return dest_dir

def stage_to_drive(stage_path: Path, drive_subdir: str):
    drive_dest = RAW / drive_subdir
    if drive_dest.exists() and any(drive_dest.iterdir()):
        print(f'  [skip] {drive_dest} already on Drive')
        return drive_dest
    drive_dest.parent.mkdir(parents=True, exist_ok=True)
    if stage_path.is_dir():
        print(f'  [cp]   {stage_path} -> {drive_dest}  (may take a minute)')
        shutil.copytree(stage_path, drive_dest, dirs_exist_ok=True)
    else:
        drive_dest.mkdir(parents=True, exist_ok=True)
        target = drive_dest / stage_path.name
        print(f'  [cp]   {stage_path} -> {target}')
        shutil.copy2(stage_path, target)
    return drive_dest

## 4. NSL-KDD

Tiny dataset (~25 MB). We download it first as a pipeline sanity check — if this fails, something is wrong with networking before we get to the big files.

In [4]:
print('=' * 60)
print('Dataset 1/4: NSL-KDD (via Kaggle)')
print('=' * 60)

stage_nsl = STAGE / 'nsl_kdd'
NSL_SLUG = 'hassan06/nslkdd'   # 96k+ downloads, stable since 2019

try:
    download_kaggle_dataset(NSL_SLUG, stage_nsl)
    stage_to_drive(stage_nsl, 'nsl_kdd')
    print('\nNSL-KDD done ✓\n')
except Exception as e:
    print(f'\n[WARN] Kaggle download failed: {e}')
    print('Did you accept terms at https://www.kaggle.com/datasets/hassan06/nslkdd ?')

Dataset 1/4: NSL-KDD (via Kaggle)
  [get]  kaggle datasets download -d hassan06/nslkdd
  [done] extracted to /content/data_stage/nsl_kdd
  [cp]   /content/data_stage/nsl_kdd -> /content/drive/MyDrive/phd_thesis/data/raw/nsl_kdd  (may take a minute)

NSL-KDD done ✓



## 5. CICIDS2017 — Original (for comparability)

The unmodified Sharafaldin/Lashkari/Ghorbani 2018 release. We don't use this for experiments, but we keep it so we can reproduce numbers from older papers that used the buggy version.

In [5]:
print('=' * 60)
print('Dataset 2/4: CICIDS2017 (original CSVs)')
print('=' * 60)

stage_cic = STAGE / 'cicids2017_original'
stage_cic.mkdir(exist_ok=True)

# Kaggle has a well-mirrored copy of the official MachineLearningCSV release.
# If the slug below ever becomes stale, alternatives: 'devendra416/ddos-datasets',
# 'cicdataset/cicids2017'. The slug below is the most-downloaded mirror.
ORIGINAL_SLUG = 'cicdataset/cicids2017'

try:
    download_kaggle_dataset(ORIGINAL_SLUG, stage_cic)
    stage_to_drive(stage_cic, 'cicids2017_original')
    print('\nCICIDS2017 original done ✓\n')
except Exception as e:
    print(f'\n[WARN] Kaggle download failed: {e}')
    print('Manual fallback:')
    print('  1. Go to https://www.unb.ca/cic/datasets/ids-2017.html')
    print('  2. Fill the request form (free, instant)')
    print('  3. Download MachineLearningCSV.zip')
    print(f'  4. Unzip into {RAW / "cicids2017_original"}')
    print('  5. Re-run this cell to verify')

Dataset 2/4: CICIDS2017 (original CSVs)
  [get]  kaggle datasets download -d cicdataset/cicids2017
  [FAIL] 

[WARN] Kaggle download failed: 
Manual fallback:
  1. Go to https://www.unb.ca/cic/datasets/ids-2017.html
  2. Fill the request form (free, instant)
  3. Download MachineLearningCSV.zip
  4. Unzip into /content/drive/MyDrive/phd_thesis/data/raw/cicids2017_original
  5. Re-run this cell to verify


## 6. CICIDS2017 — Engelen Improved (PRIMARY DATASET)

This is the cleaned version we use for experiments. Fixed:
- Duplicate flows removed
- Label errors corrected
- CICFlowMeter v3 bugs fixed (FIN-termination bug, bulk-feature bug, down/up ratio bug)
- New `X-Attempted` label class for payload-less attack attempts

Canonical source: Engelen, Rimmer, Joosen (2021) WTMC, hosted at https://intrusion-detection.distrinet-research.be/WTMC2021/

We pull it through Kaggle (`dhoogla/improved-cicids2017-and-csecicids2018`) because the canonical site doesn't permit programmatic download. **Verify after download that you have the Engelen-version columns** (check Cell 9).

In [6]:
print('=' * 60)
print('Dataset 2/3: CICIDS2017 IMPROVED (Engelen et al. 2021)')
print('=' * 60)
print('Note: this download is ~11GB (bundles improved CICIDS2017 +')
print('improved CSE-CIC-IDS2018). Will take ~5-15 minutes on Colab Pro+.')

stage_eng = STAGE / 'cicids2017_improved'
IMPROVED_SLUG = 'ernie55ernie/improved-cicids2017-and-csecicids2018'

try:
    download_kaggle_dataset(IMPROVED_SLUG, stage_eng)
    stage_to_drive(stage_eng, 'cicids2017_improved')
    print('\nCICIDS2017 improved done ✓\n')
except Exception as e:
    print(f'\n[WARN] Kaggle download failed: {e}')
    print('Did you accept terms at')
    print('  https://www.kaggle.com/datasets/ernie55ernie/improved-cicids2017-and-csecicids2018 ?')

Dataset 2/3: CICIDS2017 IMPROVED (Engelen et al. 2021)
Note: this download is ~11GB (bundles improved CICIDS2017 +
improved CSE-CIC-IDS2018). Will take ~5-15 minutes on Colab Pro+.
  [get]  kaggle datasets download -d ernie55ernie/improved-cicids2017-and-csecicids2018
  [done] extracted to /content/data_stage/cicids2017_improved
  [cp]   /content/data_stage/cicids2017_improved -> /content/drive/MyDrive/phd_thesis/data/raw/cicids2017_improved  (may take a minute)

CICIDS2017 improved done ✓



## 7. UNSW-NB15

Second real dataset for Phase 2 cross-dataset generalization.

In [7]:
print('=' * 60)
print('Dataset 4/4: UNSW-NB15')
print('=' * 60)

stage_unsw = STAGE / 'unsw_nb15'
stage_unsw.mkdir(exist_ok=True)

UNSW_SLUG = 'mrwellsdavid/unsw-nb15'

try:
    download_kaggle_dataset(UNSW_SLUG, stage_unsw)
    stage_to_drive(stage_unsw, 'unsw_nb15')
    print('\nUNSW-NB15 done ✓\n')
except Exception as e:
    print(f'\n[WARN] Kaggle download failed: {e}')
    print('Manual fallback:')
    print('  1. Go to https://research.unsw.edu.au/projects/unsw-nb15-dataset')
    print('  2. Download the CSV partitions (training-set.csv, testing-set.csv)')
    print(f'  3. Place in {RAW / "unsw_nb15"}')

Dataset 4/4: UNSW-NB15
  [get]  kaggle datasets download -d mrwellsdavid/unsw-nb15
  [done] extracted to /content/data_stage/unsw_nb15
  [cp]   /content/data_stage/unsw_nb15 -> /content/drive/MyDrive/phd_thesis/data/raw/unsw_nb15  (may take a minute)

UNSW-NB15 done ✓



## 8. Verification summary

In [8]:
print('=' * 60)
print('DOWNLOAD SUMMARY')
print('=' * 60)

def summarize(d: Path):
    if not d.exists():
        return f'  [MISSING] {d.name}'
    files = [f for f in d.rglob('*') if f.is_file() and not f.name.startswith('.')]
    total = sum(f.stat().st_size for f in files)
    return f'  {d.name:30s}  {len(files):4d} files  {human_size(total):>10s}'

for sub in sorted(RAW.iterdir()):
    if sub.is_dir():
        print(summarize(sub))

print()
disk_status()
print('\nIf any dataset shows [MISSING] or 0 files, re-run that cell or use the')
print('manual fallback printed above. Do not proceed to data cleaning until all')
print('four datasets are present.')

DOWNLOAD SUMMARY
  cicids2017_improved               15 files      34.6GB
  nsl_kdd                           22 files     106.6MB
  unsw_nb15                          8 files     604.7MB

Drive:  31.9GB free of 100.0GB
Local:  134.7GB free of 225.8GB

If any dataset shows [MISSING] or 0 files, re-run that cell or use the
manual fallback printed above. Do not proceed to data cleaning until all
four datasets are present.


## 9. Sanity-check the Engelen improved dataset

Verify that the dataset we downloaded is genuinely the Engelen-improved version (not a re-mirror of the buggy original under a misleading name). Engelen-improved should have:
- An `Attempted Category` column (their new label class for payload-less attempts) **OR** at minimum, labels including `*-Attempted` variants
- Day-named files (`monday.csv` ... `friday.csv`), not the original `Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv` naming

In [9]:
import pandas as pd
from pathlib import Path

improved_dir = Path('/content/drive/MyDrive/phd_thesis/data/raw/cicids2017_improved')

# List ALL files with their sizes
all_csvs = sorted(improved_dir.rglob('*.csv'))
print(f"All {len(all_csvs)} CSV files:\n")
for f in all_csvs:
    size_mb = f.stat().st_size / 1024**2
    print(f"  {str(f.relative_to(improved_dir)):65s} {size_mb:>10.1f} MB")

# Group by year
cic2017 = [f for f in all_csvs if '2017' in str(f).lower()]
cic2018 = [f for f in all_csvs if '2018' in str(f).lower()]
print(f"\nCICIDS2017 files: {len(cic2017)}")
print(f"CSE-CIC-IDS-2018 files: {len(cic2018)}")

# Sample CICIDS2017 Wednesday (DoS attack day — should have many attacks)
wed_2017 = [f for f in cic2017 if 'wednesday' in f.name.lower() or 'wed' in f.name.lower()]
sample = wed_2017[0] if wed_2017 else (cic2017[0] if cic2017 else all_csvs[0])

print(f"\n=== Sampling: {sample.relative_to(improved_dir)} ===")

# Read the full file this time (not just 50k rows) so we see all label types
df = pd.read_csv(sample, low_memory=False)
print(f"Shape: {df.shape}")
print(f"\nAll {len(df.columns)} columns:")
for i in range(0, len(df.columns), 4):
    chunk = df.columns[i:i+4]
    print("  " + " | ".join(f"{c[:20]:<20s}" for c in chunk))

label_col = next((c for c in df.columns if 'label' in c.lower()), None)
if label_col:
    print(f"\nFull label distribution in {sample.name}:")
    print(df[label_col].value_counts().to_string())
    has_attempted = df[label_col].astype(str).str.contains('Attempted', case=False).any()
    print(f"\n'*-Attempted' labels present: {has_attempted}")

    # Also check for Engelen-specific columns
    engelen_cols = ['Attempted Category', 'Flow ID', 'Src IP', 'Dst IP']
    found = [c for c in df.columns if c in engelen_cols]
    print(f"Engelen-signature columns found: {found}")

    if has_attempted or 'Attempted Category' in df.columns:
        print("\n✓ CONFIRMED: This is the Engelen-improved version.")
    elif len(df.columns) >= 88:
        print("\n~ LIKELY Engelen-improved (91-column schema matches),")
        print("  but no Attempted labels in this specific file.")
        print("  Check another day's file to be certain.")
    else:
        print("\n⚠ Column count too low — likely the original, not improved.")

All 15 CSV files:

  CICIDS2017_improved/friday.csv                                         272.0 MB
  CICIDS2017_improved/monday.csv                                         198.2 MB
  CICIDS2017_improved/thursday.csv                                       180.7 MB
  CICIDS2017_improved/tuesday.csv                                        170.1 MB
  CICIDS2017_improved/wednesday.csv                                      277.8 MB
  CSECICIDS2018_improved/Friday-02-03-2018.csv                          3518.9 MB
  CSECICIDS2018_improved/Friday-16-02-2018.csv                          4016.6 MB
  CSECICIDS2018_improved/Friday-23-02-2018.csv                          3251.4 MB
  CSECICIDS2018_improved/Thursday-01-03-2018.csv                        3632.0 MB
  CSECICIDS2018_improved/Thursday-15-02-2018.csv                        2853.9 MB
  CSECICIDS2018_improved/Thursday-22-02-2018.csv                        3312.9 MB
  CSECICIDS2018_improved/Tuesday-20-02-2018.csv                         3272.3 

In [10]:
import subprocess

print("=== Searching for CICIDS datasets on Kaggle ===\n")
result = subprocess.run(
    ['kaggle', 'datasets', 'list', '-s', 'cicids2017'],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

print("\n=== Searching for NSL-KDD ===\n")
result = subprocess.run(
    ['kaggle', 'datasets', 'list', '-s', 'nsl-kdd'],
    capture_output=True, text=True
)
print(result.stdout)

print("\n=== Searching dhoogla's datasets ===\n")
result = subprocess.run(
    ['kaggle', 'datasets', 'list', '-m', '--user', 'dhoogla'],
    capture_output=True, text=True
)
print(result.stdout)

=== Searching for CICIDS datasets on Kaggle ===

ref                                                      title                                              size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-------------------------------------------------------  ------------------------------------------  -----------  --------------------------  -------------  ---------  ---------------  
ericanacletoribeiro/cicids2017-cleaned-and-preprocessed  CICIDS2017: Cleaned & Preprocessed            210143955  2025-01-12 12:02:31.110000           7553         54                1  
ernie55ernie/improved-cicids2017-and-csecicids2018       Improved CICIDS2017 and CSECICIDS2018       10985642855  2023-08-15 16:57:44.847000           2147         21        0.8235294  
devendra416/ddos-datasets                                DDoS Dataset                                 2880543869  2019-04-30 13:49:22.867000          13657        152        0.7058824  
mdalamintalukder/cici

In [11]:
from pathlib import Path

RAW = Path('/content/drive/MyDrive/phd_thesis/data/raw')
STAGE = Path('/content/data_stage')

def show(root: Path, label: str):
    print(f"\n=== {label} ({root}) ===")
    if not root.exists():
        print("  (does not exist)")
        return
    for d in sorted(root.iterdir()):
        if d.is_dir():
            files = [f for f in d.rglob('*') if f.is_file()]
            total = sum(f.stat().st_size for f in files)
            print(f"  {d.name:35s} {len(files):5d} files  {total/1024**2:>10.1f} MB")

show(RAW, "DRIVE  (data/raw/)")
show(STAGE, "LOCAL  (/content/data_stage/)")


=== DRIVE  (data/raw/) (/content/drive/MyDrive/phd_thesis/data/raw) ===
  cicids2017_improved                    15 files     35473.6 MB
  nsl_kdd                                22 files       106.6 MB
  unsw_nb15                               8 files       604.7 MB

=== LOCAL  (/content/data_stage/) (/content/data_stage) ===
  cicids2017_improved                    15 files     35473.6 MB
  cicids2017_original                     0 files         0.0 MB
  nsl_kdd                                22 files       106.6 MB
  unsw_nb15                               8 files       604.7 MB


In [12]:
import shutil
shutil.rmtree('/content/data_stage', ignore_errors=True)
print("Local stage cleared.")

Local stage cleared.


In [13]:
from pathlib import Path
import datetime, hashlib

ROOT = Path('/content/drive/MyDrive/phd_thesis')
card_path = ROOT / 'reports' / 'advisor_notes' / 'data_card.md'
card_path.parent.mkdir(parents=True, exist_ok=True)

content = f"""# Data Card — Phase 0 Datasets

**Generated:** {datetime.date.today().isoformat()}
**Drive root:** `{ROOT}`

## Datasets in scope

### 1. CICIDS2017 (Engelen-improved) — PRIMARY

- **Source:** Kaggle mirror `ernie55ernie/improved-cicids2017-and-csecicids2018`
- **Canonical origin:** Engelen, Rimmer, Joosen (2021), WTMC, Distrinet-KU Leuven
  https://intrusion-detection.distrinet-research.be/WTMC2021/
- **Local path:** `data/raw/cicids2017_improved/CICIDS2017_improved/`
- **Files:** 5 day-named CSVs (monday.csv ... friday.csv)
- **Size:** ~1.1 GB combined
- **Columns:** 91 (original CICIDS2017 had 79–85)
- **Label column:** `Label`
- **Attack-category column:** `Attempted Category` (Engelen-specific)
- **Verified Engelen-signature:** `*-Attempted` labels present, `Attempted Category` column present
- **Citation:**
  Engelen, G., Rimmer, V., Joosen, W. (2021). Troubleshooting an Intrusion
  Detection Dataset: the CICIDS2017 Case Study. IEEE Security and Privacy
  Workshops (SPW), pp. 7–12.
- **Use in thesis:** Phase 1 replication, Phase 2 primary dataset, Phase 3 primary dataset

### 2. CSE-CIC-IDS-2018 (Engelen-improved) — RESERVE

- **Source:** Same Kaggle dataset bundle
- **Local path:** `data/raw/cicids2017_improved/CSECICIDS2018_improved/`
- **Files:** 10 date-named CSVs (Feb 14, 2018 → Mar 2, 2018)
- **Size:** ~34 GB combined
- **Use in thesis:** Reserve for Phase 2 cross-dataset generalization (if needed)

### 3. UNSW-NB15

- **Source:** Kaggle `mrwellsdavid/unsw-nb15`
- **Local path:** `data/raw/unsw_nb15/`
- **Size:** ~605 MB
- **Citation:**
  Moustafa, N., Slay, J. (2015). UNSW-NB15: A Comprehensive Data Set for
  Network Intrusion Detection Systems. MilCIS, IEEE.
- **Use in thesis:** Phase 2 cross-dataset generalization (primary second dataset)

### 4. NSL-KDD

- **Source:** Kaggle `hassan06/nslkdd`
- **Local path:** `data/raw/nsl_kdd/`
- **Size:** ~107 MB
- **Citation:**
  Tavallaee, M., Bagheri, E., Lu, W., Ghorbani, A. A. (2009). A Detailed
  Analysis of the KDD CUP 99 Data Set. IEEE Symposium on Computational
  Intelligence for Security and Defense Applications.
- **Use in thesis:** Pipeline sanity check, debugging on small data

## Known issues / preprocessing required

- All CICIDS-style datasets have `inf` and `NaN` values in some flow-rate columns
  (Flow Bytes/s, Flow Packets/s when duration is zero). Drop or impute before training.
- Class imbalance is severe across all datasets (benign typically >> attack).
- Some columns have leading or trailing whitespace in headers — strip before use.

## Licenses / terms of use

- CICIDS2017: see UNB CIC dataset terms — academic use permitted, citation required
- Engelen improved version: same terms as original, additional Engelen 2021 citation required
- UNSW-NB15: academic use permitted, citation required
- NSL-KDD: open academic use
"""

card_path.write_text(content)
print(f'Written: {card_path}')
print(f'Size: {card_path.stat().st_size} bytes')

Written: /content/drive/MyDrive/phd_thesis/reports/advisor_notes/data_card.md
Size: 2862 bytes


## 10. Required reading and citations

Before running any experiments on these datasets, read these papers. Each one will save you hours of confusion later.

### Critical (read this week)

**Engelen, G., Rimmer, V., Joosen, W. (2021).** *Troubleshooting an Intrusion Detection Dataset: the CICIDS2017 Case Study.* In 2021 IEEE Security and Privacy Workshops (SPW), pp. 7–12.

- PDF: https://intrusion-detection.distrinet-research.be/WTMC2021/Resources/wtmc2021_Engelen_Troubleshooting.pdf
- Project page (extended docs): https://intrusion-detection.distrinet-research.be/WTMC2021/
- Code/tool repository: https://github.com/GintsEngelen/WTMC2021-Code
- Why: documents exactly what was wrong with the original CICIDS2017 and what the improved version fixes. You must understand this or reviewers will catch you.

### Foundational dataset citations (cite in methods chapter)

**Sharafaldin, I., Lashkari, A. H., Ghorbani, A. A. (2018).** *Toward Generating a New Intrusion Detection Dataset and Intrusion Traffic Characterization.* ICISSP. — original CICIDS2017.

**Moustafa, N., Slay, J. (2015).** *UNSW-NB15: A Comprehensive Data Set for Network Intrusion Detection Systems.* Military Communications and Information Systems Conference (MilCIS), IEEE. — UNSW-NB15.

**Tavallaee, M., Bagheri, E., Lu, W., Ghorbani, A. A. (2009).** *A Detailed Analysis of the KDD CUP 99 Data Set.* IEEE Symposium on Computational Intelligence for Security and Defense Applications. — NSL-KDD.

### Optional but useful

**Liu, L., Engelen, G., Lynar, T., Essam, D., Joosen, W. (2022).** *Error Prevalence in NIDS datasets: A Case Study on CIC-IDS-2017 and CSE-CIC-IDS-2018.* IEEE CNS 2022. — deeper analysis of dataset issues; bridges the 2021 Engelen paper to subsequent work.

## 11. Next steps

Once all four datasets show ✓ in the summary:

1. **Read the Engelen paper** before writing any data loader. It tells you which columns to trust and which to drop.
2. **Write `src/data/loaders.py`** — unified loader interface with separate functions for each dataset. The next notebook (`03_data_characterization.ipynb`) will use it.
3. **Write a one-page data card** in `reports/advisor_notes/data_card.md` recording: source URL, download date, file count, total size, key columns, license. This becomes part of your thesis methods chapter.
4. **Commit the notebook** (not the data — `.gitignore` already excludes `data/raw/`).